In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import matplotlib
import seaborn as sns
import copy
import pickle
import warnings
warnings.filterwarnings('ignore')
import shap
from models import *
from utilities import *

In [35]:
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42 
matplotlib.rc('font', family='sans-serif') 
matplotlib.rc('font', serif='Helvetica Neue') 

In [36]:
##########################################################################################################

In [37]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("Eager mode:", tf.executing_eagerly())
print("GPUs:", tf.config.list_physical_devices('GPU'))

TF version: 2.15.0
Eager mode: True
GPUs: []


In [38]:
# shap.explainers._deep.deep_tf.op_handlers["AddV2"] = shap.explainers._deep.deep_tf.passthrough 
# shap.explainers._deep.deep_tf.op_handlers["FusedBatchNormV3"] = shap.explainers._deep.deep_tf.passthrough

In [39]:
################### data import ###############

In [40]:
data_dir = '../results/data/'
data_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_dy_filter.pkl')
ba_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_ba_filter.pkl')

In [41]:
data_copy['Id_encoded'], uniques = pd.factorize(data_copy['SCCRIP_ID'])
mapping_dict = {value: index for index, value in enumerate(uniques)}

In [42]:
ba_copy['Id_encoded'] = ba_copy['SCCRIP_ID'].map(mapping_dict)
ba_copy = ba_copy[ba_copy['Id_encoded'].notna()]
ba_copy = ba_copy.drop_duplicates(subset='SCCRIP_ID', keep='last')
ba_copy.drop('SCCRIP_ID', inplace=True, axis=1)

In [43]:
ba_copy = ba_copy.fillna(0)
data_copy = data_copy.fillna(0)

bmt = read_sas('../data/Apply_MI_Zheng_2023_01/othertreatment_summary.sas7bdat')
bmt = bmt[['SCCRIP_ID', 'BMT']]
bmt_ids = set(bmt[bmt['BMT'] == 1].SCCRIP_ID)
data_copy_filter = data_copy[~data_copy["SCCRIP_ID"].isin(bmt_ids)]
len(data_copy_filter.SCCRIP_ID.unique())

1264

In [44]:
feature_dy = data_copy_filter.drop(['HU_status','Id_encoded','SCCRIP_ID', 'date'],axis=1).columns
feature_ba = ba_copy.drop(['Id_encoded'], axis=1).columns

In [45]:
###############################################

In [46]:
seq_length = 5
r2 = []
shap_impact = []
sequences, seq_ba, labels, date,  y_target = create_dataset(data_copy_filter, ba_copy, seq_length, mapping_dict)
sequences_copy, seq_ba_copy, labels_copy, date_copy, y_target_copy = shuffle_dataset(sequences, seq_ba, labels, date, y_target)
X, y, X_bg, dt, ls, seq_map = create_sequences_with_background(sequences_copy, labels_copy, seq_ba_copy, date_copy,y_target_copy, seq_length)
X_normalized, X_bg_normalized = normalize_X(X, X_bg)
while len(r2) < 20:
    print (len(r2))
    (X_train,X_val,X_test,
                X_bg_train,X_bg_val,X_bg_test,
                dt_train, dt_val, dt_test, 
                y_train,y_val,y_test,
                ls_train, ls_val, ls_test,
                seq_map_train, seq_map_val, seq_map_test) = split_dataset(X_normalized, X_bg_normalized,dt, y, ls, seq_map)
    
    combined_model_l = CombinedLSTMModel(seq_length=seq_length, n_features=int(X_train.shape[2]), bg_features=int(X_bg_train.shape[2]), summary=False)
    
    history = combined_model_l.train(
    X_train, X_bg_train, dt_train, y_train,
    X_val,   X_bg_val,   dt_val,   y_val,
    epochs=20, batch_size=64,
    checkpoint_path="../results/models/Mix_best2_loop.h5")
    y_pred = combined_model_l.predict(X_test, X_bg_test, dt_test)
    r2_ = r2_score(y_test.flatten(), y_pred.flatten())
    if r2_ > 0:
        r2.append(r2_)

        explainer = shap.GradientExplainer(combined_model_l.model, [X_test, X_bg_test, dt_test])
        shap_values = explainer.shap_values([X_test, X_bg_test, dt_test])

        shap_val_d = []
        train_d = []

        for i in range (5):
            shap_val_d.append(pd.DataFrame(shap_values[0][:,i,:,0], columns=feature_dy))
            train_d.append(pd.DataFrame(X_test[:,i,:],columns=feature_dy))
            
        shap_val_b = pd.DataFrame(shap_values[1][:,0,:,0], columns=feature_ba)
        train_b = pd.DataFrame(X_bg_test[:,0,:], columns=feature_ba)

        impact_dy = []

        for i in range(5):
            im = ABS_SHAP(shap_val_d[i], train_d[i])
            impact_dy.append(im)
            
        impact_ba = ABS_SHAP(shap_val_b, train_b)

        shap_impact.append((impact_dy, impact_ba))

0
Epoch 1/20
295/295 [==============================] - ETA: 0s - loss: 64.3546 - mean_absolute_error: 4.9095
Epoch 1: val_loss improved from inf to 23.74369, saving model to ../results/models/Mix_best2_loop.h5
295/295 [==============================] - 4s 7ms/step - loss: 64.3546 - mean_absolute_error: 4.9095 - val_loss: 23.7437 - val_mean_absolute_error: 3.1492
Epoch 2/20
285/295 [===========================>..] - ETA: 0s - loss: 14.7252 - mean_absolute_error: 2.5828
Epoch 2: val_loss improved from 23.74369 to 17.29431, saving model to ../results/models/Mix_best2_loop.h5
295/295 [==============================] - 2s 6ms/step - loss: 14.7084 - mean_absolute_error: 2.5847 - val_loss: 17.2943 - val_mean_absolute_error: 2.7063
Epoch 3/20
286/295 [============================>.] - ETA: 0s - loss: 13.6108 - mean_absolute_error: 2.4222
Epoch 3: val_loss improved from 17.29431 to 15.67139, saving model to ../results/models/Mix_best2_loop.h5
295/295 [==============================] - 2s 6ms/s

In [47]:
with open('../results/data/mix/shap_impact_list_noBMTv5.pkl','wb') as f:
    pickle.dump(shap_impact,f)

In [48]:
with open('../results/data/mix/r2_Mixmodel_noBMT.pkl','wb') as f:
    pickle.dump(r2,f)